In [1]:
import json
import os
import re

# Compaction Analysis

In [2]:
baseline_file = r'./artifacts/django__django-15368-baseline-trajectory.json'
compaction_file = r'./artifacts/django__django-15368-trajectory.json'

In [3]:
with open(baseline_file, 'r') as f:
  baseline = json.load(f)

with open(compaction_file, 'r') as f:
  compaction = json.load(f)

In [4]:
baseline_tokens = []
for m in baseline['responses']:
  baseline_tokens.append(m['usage']['total_tokens'])

compaction_tokens = []
for m in compaction['responses']:
  compaction_tokens.append(m['usage']['total_tokens'])
  

In [17]:
baseline_tokens

[1570,
 3653,
 6203,
 7567,
 8092,
 8330,
 10147,
 11835,
 14502,
 14409,
 14990,
 15218,
 16271,
 17007]

In [20]:
compaction_tokens

[1542,
 3278,
 5583,
 5268,
 3291,
 6475,
 5958,
 2795,
 3012,
 4364,
 4452,
 5339,
 4016,
 6225,
 3317,
 6068,
 2959,
 3895,
 5392,
 2955,
 3304,
 4430,
 5721,
 3396,
 5537,
 3852,
 5366,
 5265,
 5002,
 5761,
 2517,
 4672,
 5209,
 5504,
 5876,
 6014,
 2797,
 3495]

In [ ]:
def get_run_info(record_path, result_path):
  with open(record_path, 'r') as f:
    records = json.load(f)

  with open(result_path, 'r') as f:
    results = json.load(f)

  game_over = eval(re.search('<chess_state>.+\ngame_over:(.*?)\n.+</chess_state>', results, re.DOTALL).group(1).strip())

  trajectory = records['prompts'][-1]
  play_move_call_ids = set()
  for info in trajectory:
    if info.get('role', '') == 'assistant':
      for call in info.get('tool_calls', []):
        if call.get('function', {}).get('name', '') == 'play_move':
          play_move_call_ids.add(call.get('id', ''))

  failed_play_moves = 0
  for info in trajectory:
    if info.get('role', '') == 'tool':
      if info.get('tool_call_id', '') in play_move_call_ids:
        if '<chess_error>' in info.get('content', ''):
          failed_play_moves += 1

  total_play_moves = len(play_move_call_ids)

  return total_play_moves + 1, failed_play_moves, game_over


In [10]:
get_run_info('artifacts/part3-no-legal-moves-deepseek.json', 'artifacts/part3-no-legal-moves-deepseek-result.json')

(62, 12, True)

In [11]:
get_run_info('artifacts/part3-legal-moves-deepseek.json', 'artifacts/part3-legal-moves-deepseek-result.json')

(18, 0, True)

In [12]:
get_run_info('artifacts/part3-no-legal-moves-gpt-oss.json', 'artifacts/part3-no-legal-moves-gpt-oss-result.json')

(18, 0, True)

In [13]:
get_run_info('artifacts/part3-legal-moves-gpt-oss.json', 'artifacts/part3-legal-moves-gpt-oss-result.json')

(19, 0, True)